Aggregate per-year storm exceedance metrics (from `storm_percentile_metrics.py`) over the full 35-year period for each simulation. Storms spanning two calendar years are summed.

In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [2]:
simulations = ['UBD', 'UBE', 'UBF', 'UBG', 'UBH', 'UBI']
future_hist_sim = {'UBG': 'UBD', 'UBH': 'UBE', 'UBI': 'UBF'}
wetdays = False
future = False

In [3]:
base = '/home/vdemeyer/projects/rrg-gachon/vdemeyer'

for sim in simulations:

    # Determine period
    if sim in future_hist_sim:
        years = range(2063, 2098)
    else:
        years = range(1980, 2015)

    # Build suffix for filenames
    add_file = ''
    if wetdays:
        add_file += '_wetdays'
    if future and sim in future_hist_sim:
        add_file += '_future_percentile'

    # Load all per-year pickles
    dfs = []
    for year in tqdm(years, desc=sim):
        pkl_file = f'{base}/TRACKING/KATJA/OUTPUTS/{sim}/STORM_METRICS/storm_exceed_999_metrics_landonly_{sim}_{year}{add_file}.pkl'
        if not os.path.exists(pkl_file):
            print(f"Warning: missing {pkl_file}")
            continue
        dfs.append(pd.read_pickle(pkl_file))

    if not dfs:
        print(f"No pickle files found for {sim}, skipping")
        continue

    # Concatenate and sum cum/count for storms appearing in multiple years
    df_all = pd.concat(dfs, ignore_index=True)
    df_agg = df_all.groupby('storm_id', as_index=False).agg({
        'cum_excess_pr': 'sum',
        'count_exceed_pr': 'sum',
        'cum_excess_wind': 'sum',
        'count_exceed_wind': 'sum',
    })

    # Compute intensity = cum / count
    df_agg['mean_excess_pr'] = np.where(
        df_agg['count_exceed_pr'] > 0,
        df_agg['cum_excess_pr'] / df_agg['count_exceed_pr'],
        np.nan
    )
    df_agg['mean_excess_wind'] = np.where(
        df_agg['count_exceed_wind'] > 0,
        df_agg['cum_excess_wind'] / df_agg['count_exceed_wind'],
        np.nan
    )

    # Load tracking data and extract initiation date per storm
    storm_data_file = f'{base}/TRACKING/KATJA/OUTPUTS/{sim}_psl_smooth_400km_12h_1000hPa.txt'
    df_track = pd.read_csv(
        storm_data_file,
        sep=r' ', header=0, engine='python',
        names=['storm', 'point', 'i', 'j', 'date', 'lat', 'lon', 'pressure']
    )
    df_track['date'] = pd.to_datetime(df_track['date'])
    initiation = df_track.groupby('storm')['date'].min().reset_index()
    initiation.columns = ['storm_id', 'initiation_date']

    # Merge initiation date
    df_agg = df_agg.merge(initiation, on='storm_id', how='left')

    # Filter: keep only storms whose initiation falls within the period
    start_year, end_year = min(years), max(years)
    df_agg = df_agg[
        (df_agg['initiation_date'].dt.year >= start_year) &
        (df_agg['initiation_date'].dt.year <= end_year)
    ]

    # Reorder columns
    df_agg = df_agg[['storm_id', 'initiation_date',
                      'cum_excess_pr', 'count_exceed_pr', 'mean_excess_pr',
                      'cum_excess_wind', 'count_exceed_wind', 'mean_excess_wind']]
    df_agg = df_agg.sort_values('storm_id').reset_index(drop=True)

    # Save
    output_dir = f'{base}/TRACKING/KATJA/OUTPUTS/{sim}/STORM_METRICS/'
    os.makedirs(output_dir, exist_ok=True)
    output_file = f'{output_dir}/storm_exceed_999_metrics_landonly_{sim}_{start_year}-{end_year}{add_file}.pkl'
    df_agg.to_pickle(output_file)
    print(f"\n{sim}: saved {len(df_agg)} storms to {output_file}")
    print(f"  Storms with pr exceedance: {(df_agg['count_exceed_pr'] > 0).sum()}")
    print(f"  Storms with wind exceedance: {(df_agg['count_exceed_wind'] > 0).sum()}")

UBD: 100%|██████████| 35/35 [00:01<00:00, 18.93it/s]



UBD: saved 10608 storms to /home/vdemeyer/projects/rrg-gachon/vdemeyer/TRACKING/KATJA/OUTPUTS/UBD/STORM_METRICS//storm_exceed_999_metrics_landonly_UBD_1980-2014.pkl
  Storms with pr exceedance: 9025
  Storms with wind exceedance: 7825


UBE: 100%|██████████| 35/35 [00:01<00:00, 33.59it/s]



UBE: saved 10280 storms to /home/vdemeyer/projects/rrg-gachon/vdemeyer/TRACKING/KATJA/OUTPUTS/UBE/STORM_METRICS//storm_exceed_999_metrics_landonly_UBE_1980-2014.pkl
  Storms with pr exceedance: 8761
  Storms with wind exceedance: 7700


UBF: 100%|██████████| 35/35 [00:01<00:00, 34.04it/s]


...


UBG: saved 9730 storms to /home/vdemeyer/projects/rrg-gachon/vdemeyer/TRACKING/KATJA/OUTPUTS/UBG/STORM_METRICS//storm_exceed_999_metrics_landonly_UBG_2063-2097.pkl
  Storms with pr exceedance: 8739
  Storms with wind exceedance: 7314


UBH: 100%|██████████| 35/35 [00:03<00:00, 11.29it/s]



UBH: saved 10005 storms to /home/vdemeyer/projects/rrg-gachon/vdemeyer/TRACKING/KATJA/OUTPUTS/UBH/STORM_METRICS//storm_exceed_999_metrics_landonly_UBH_2063-2097.pkl
  Storms with pr exceedance: 8823
  Storms with wind exceedance: 7526


UBI: 100%|██████████| 35/35 [00:01<00:00, 19.87it/s]



UBI: saved 11010 storms to /home/vdemeyer/projects/rrg-gachon/vdemeyer/TRACKING/KATJA/OUTPUTS/UBI/STORM_METRICS//storm_exceed_999_metrics_landonly_UBI_2063-2097.pkl
  Storms with pr exceedance: 9715
  Storms with wind exceedance: 8194
